In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import models
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F

In [2]:
print("=" * 70)
print("FINISHING EVALUATION")
print("=" * 70)

FINISHING EVALUATION


In [3]:
CONFIG = {
    'metadata_csv': 'processed_mels/all_metadata.csv',
    'target_frames': 256,
    'batch_size': 32,
    'num_workers': 0,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'random_seed': 42,
}

In [4]:
results_dir = Path('results')
results_dir.mkdir(parents=True, exist_ok=True)

In [5]:
class MelSegmentDataset(Dataset):
    def __init__(self, metadata_csv, target_frames=256):
        self.df = pd.read_csv(metadata_csv)
        self.paths = self.df['file'].tolist()
        self.labels = self.df['label'].tolist()
        
        classes = sorted(set(self.labels))
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}
        self.y = [self.class_to_idx[l] for l in self.labels]
        self.target_frames = target_frames
        
    def _pad_or_crop(self, mel):
        n_mels, T = mel.shape
        if T == self.target_frames:
            return mel
        if T > self.target_frames:
            start = (T - self.target_frames) // 2
            return mel[:, start:start + self.target_frames]
        return np.pad(mel, ((0, 0), (0, self.target_frames - T)), mode='constant')
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        mel = np.load(self.paths[idx])
        mel = self._pad_or_crop(mel)
        mel = mel[np.newaxis, :, :]
        return torch.from_numpy(mel).float(), self.y[idx]

In [6]:
def patient_independent_split(metadata_csv, train_ratio=0.7, val_ratio=0.15, random_seed=42):
    df = pd.read_csv(metadata_csv)
    df['patient_id'] = df['audio_file'].apply(lambda x: Path(x).parent.name)
    
    patients = df['patient_id'].unique()
    n_patients = len(patients)
    n_train = int(n_patients * train_ratio)
    n_val = int(n_patients * val_ratio)
    
    np.random.seed(random_seed)
    np.random.shuffle(patients)
    
    train_patients = patients[:n_train]
    val_patients = patients[n_train:n_train+n_val]
    test_patients = patients[n_train+n_val:]
    
    train_idx = df[df['patient_id'].isin(train_patients)].index.tolist()
    val_idx = df[df['patient_id'].isin(val_patients)].index.tolist()
    test_idx = df[df['patient_id'].isin(test_patients)].index.tolist()
    
    return train_idx, val_idx, test_idx

In [7]:
def create_resnet_model(num_classes):
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    
    old_conv = model.conv1
    model.conv1 = nn.Conv2d(1, old_conv.out_channels, 
                           kernel_size=old_conv.kernel_size,
                           stride=old_conv.stride,
                           padding=old_conv.padding,
                           bias=old_conv.bias is not None)
    
    with torch.no_grad():
        model.conv1.weight[:] = old_conv.weight.mean(dim=1, keepdim=True)
    
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

print("\n[1/3] Loading data and model...")
torch.manual_seed(CONFIG['random_seed'])
np.random.seed(CONFIG['random_seed'])

# Load data
_, _, test_idx = patient_independent_split(CONFIG['metadata_csv'], random_seed=CONFIG['random_seed'])
dataset = MelSegmentDataset(CONFIG['metadata_csv'], target_frames=CONFIG['target_frames'])
test_ds = Subset(dataset, test_idx)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

# Create and load model
num_classes = len(dataset.class_to_idx)
model = create_resnet_model(num_classes).to(CONFIG['device'])

checkpoint_path = 'best_model.pth'
if not Path(checkpoint_path).exists():
    print(f" ERROR: {checkpoint_path} not found!")
    exit(1)

# Load with PyTorch 2.6+ compatibility
checkpoint = torch.load(checkpoint_path, map_location=CONFIG['device'], weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"✓ Loaded model from epoch {checkpoint['epoch']+1}")
print(f"✓ Best validation F1: {checkpoint['val_f1']:.3f}")

print("\n[2/3] Evaluating on test set...")



[1/3] Loading data and model...
✓ Loaded model from epoch 1
✓ Best validation F1: 0.760

[2/3] Evaluating on test set...


In [8]:
model.eval()
all_preds, all_true, all_probs = [], [], []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(CONFIG['device'])
        outputs = model(X)
        probs = F.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(y.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_true = np.array(all_true)
all_probs = np.array(all_probs)

class_names = [dataset.idx_to_class[i] for i in range(num_classes)]

# Classification Report
print("\n" + "=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)
report = classification_report(all_true, all_preds, target_names=class_names, digits=3)
print(report)

with open('results/classification_report.txt', 'w') as f:
    f.write(report)

# Confusion Matrix
cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.title('Confusion Matrix - Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# Per-Class Metrics
precision, recall, f1, support = precision_recall_fscore_support(all_true, all_preds, average=None)

print("\n" + "=" * 70)
print("PER-CLASS METRICS")
print("=" * 70)
print(f"{'Class':<15} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support':<10}")
print("-" * 70)
for i, name in enumerate(class_names):
    print(f"{name:<15} {precision[i]:<12.3f} {recall[i]:<12.3f} {f1[i]:<12.3f} {support[i]:<10}")

# Overall Metrics
accuracy = np.mean(all_preds == all_true)
macro_f1 = f1_score(all_true, all_preds, average='macro')
weighted_f1 = f1_score(all_true, all_preds, average='weighted')

print("\n" + "=" * 70)
print("OVERALL TEST SET METRICS")
print("=" * 70)
print(f"Accuracy:     {accuracy:.3f}")
print(f"Macro F1:     {macro_f1:.3f}")
print(f"Weighted F1:  {weighted_f1:.3f}")
print("=" * 70)



CLASSIFICATION REPORT
              precision    recall  f1-score   support

       apnea      0.807     0.678     0.737      3570
      normal      0.687     0.813     0.745      3103

    accuracy                          0.741      6673
   macro avg      0.747     0.746     0.741      6673
weighted avg      0.751     0.741     0.741      6673


PER-CLASS METRICS
Class           Precision    Recall       F1-Score     Support   
----------------------------------------------------------------------
apnea           0.807        0.678        0.737        3570      
normal          0.687        0.813        0.745        3103      

OVERALL TEST SET METRICS
Accuracy:     0.741
Macro F1:     0.741
Weighted F1:  0.741


In [9]:
results = {
    'accuracy': accuracy,
    'macro_f1': macro_f1,
    'weighted_f1': weighted_f1,
    'per_class_precision': precision,
    'per_class_recall': recall,
    'per_class_f1': f1,
    'confusion_matrix': cm,
    'predictions': all_preds,
    'true_labels': all_true,
    'probabilities': all_probs,
}

np.save('results/results.npy', results)

print("\n[3/3] Creating training history plot...")

if 'history' in checkpoint:
    history = checkpoint['history']
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss
    axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0, 0].set_xlabel('Epoch', fontsize=11)
    axes[0, 0].set_ylabel('Loss', fontsize=11)
    axes[0, 0].set_title('Training Loss', fontsize=12, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(history['train_acc'], label='Train Acc', linewidth=2)
    axes[0, 1].plot(history['val_acc'], label='Val Acc', linewidth=2)
    axes[0, 1].set_xlabel('Epoch', fontsize=11)
    axes[0, 1].set_ylabel('Accuracy', fontsize=11)
    axes[0, 1].set_title('Accuracy', fontsize=12, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].axhline(y=0.76, color='green', linestyle='--', alpha=0.5, label='Best Val')
    
    # F1-Score
    axes[1, 0].plot(history['val_f1'], label='Val F1', color='green', linewidth=2)
    axes[1, 0].set_xlabel('Epoch', fontsize=11)
    axes[1, 0].set_ylabel('F1-Score', fontsize=11)
    axes[1, 0].set_title('Validation F1-Score', fontsize=12, fontweight='bold')
    axes[1, 0].axhline(y=0.76, color='red', linestyle='--', alpha=0.5, label='Best (Epoch 1)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[1, 1].plot(history['lr'], label='Learning Rate', color='orange', linewidth=2)
    axes[1, 1].set_xlabel('Epoch', fontsize=11)
    axes[1, 1].set_ylabel('Learning Rate', fontsize=11)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
    axes[1, 1].set_yscale('log')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('results/training_history.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✓ Training history plot saved")

print("\n" + "=" * 70)
print(" EVALUATION COMPLETE!")
print("=" * 70)
print("\nResults saved in 'results/' directory:")
print("  ✓ classification_report.txt")
print("  ✓ confusion_matrix.png")
print("  ✓ training_history.png")
print("  ✓ results.npy")
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Best Model: Epoch 1")
print(f"Validation F1: 76.0%")
print(f"Test Accuracy: {accuracy:.1%}")
print(f"Test F1: {macro_f1:.1%}")
print("=" * 70)



[3/3] Creating training history plot...
✓ Training history plot saved

 EVALUATION COMPLETE!

Results saved in 'results/' directory:
  ✓ classification_report.txt
  ✓ confusion_matrix.png
  ✓ training_history.png
  ✓ results.npy

SUMMARY
Best Model: Epoch 1
Validation F1: 76.0%
Test Accuracy: 74.1%
Test F1: 74.1%
